# 20 · LLM batch runs
Runs `run_inference.py` for every (model, task, split[, variant]) job in `CONFIG`, one subprocess per job: each job loads its model fresh, so GPU memory is freed and one crash doesn't stop the batch.
- **Inputs:** `configs/llms.yaml` (15 models, pinned revisions), `data/processed/{dev,test}` (test is label-blind: no label columns, no metrics).
- **Outputs:** one experiment folder per batch, `OUTPUT_ROOT/experiments/{YYYYMMDD-HHMMSS}_experiment_{name}/`, holding `experiment.json` (config, git commit, purpose), `runs/{run_name}/` (one folder per job, see `run_inference.py`), `manifest.csv` (one row per run, rebuilt from the folders), `logs/`, per-session `status_*.csv` and a live `heartbeat.json`. Final evaluation (`99`) reads runs only through the manifest.
- **Sanity first:** with `CONFIG["sample"] = N` every job runs N random posts through the whole pipeline into a separate `{name}-sample` experiment, which evaluation refuses. Run it, check **SANITY OK**, then set `sample` to `None` and launch the full batch.
- **Background runs:** progress is in `heartbeat.json` on Drive (updated every `heartbeat_s` seconds), logs in `logs/`. Rerunning with the same `experiment` name resumes it: finished jobs are skipped, partial ones resume. A changed setting is refused; use a new name.
- `dry_run: True` only prints the job table.

In [1]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"  # on Drive, so runs survive disconnects
# On Colab, weights go to local disk: Drive is slow for multi-GB reads and 15 models need ~150 GB
HF_HOME = Path("/content/hf_cache") if IN_COLAB else PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)
sys.path.insert(0, str(PROJECT_ROOT))

# Copy the processed parquet files to local disk for speed; outputs still go to Drive
if IN_COLAB:
    LOCAL_DATA = Path("/content/data")
    shutil.copytree(DATA_ROOT / "processed", LOCAL_DATA / "processed", dirs_exist_ok=True)
    DATA_ROOT = LOCAL_DATA
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT  ", DATA_ROOT, "\nOUTPUT_ROOT", OUTPUT_ROOT, "\nHF_HOME    ", HF_HOME)

DATA_ROOT   /content/data 
OUTPUT_ROOT /content/drive/MyDrive/Sexism-research/outputs 
HF_HOME     /content/hf_cache


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
# Same pins as 00 and requirements.txt; torch is Colab's build, logged not pinned
PINS = {
    "pandas": "3.0.6", "numpy": "2.5.4", "scikit-learn": "1.9.1", "pyarrow": "26.0.0", "pyyaml": "6.0.3",
    "transformers": "5.20.1", "pydantic": "2.13.5", "outlines": "1.3.3", "xgrammar": "0.2.8",
}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS}
    print("If pip changed anything, restart the runtime, then run all again")

from importlib.metadata import version

off_pin = {p: version(p) for p in PINS if version(p) != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"

If pip changed anything, restart the runtime, then run all again


In [ ]:
import datetime

# The only cell to edit between batches; one batch = one experiment folder (src/utils/experiments.py)
CONFIG = {
    "experiment": "zeroshot-main",  # experiments/{YYYYMMDD-HHMMSS}_experiment_{name}; the same name resumes it
    "purpose": "TODO: one line on what this batch is for",
    "protocol_ref": "docs/12-protocol-addendum.md",  # the pre-registered section this batch runs
    # Sanity mode: N random posts per job into a separate '-sample' experiment; None = the full batch
    "sample": 5,
    "models": "all",  # "all" = every model in configs/llms.yaml, or a list of short names
    "tasks": [
        ("cmsb", "binary"),
        ("edos", "task_a"), ("edos", "task_b"), ("edos", "task_c"),
        ("exist", "task_1_1"), ("exist", "task_1_2"), ("exist", "task_1_3"),
    ],
    "splits": ["test"],
    "backend": "logprob",
    # Run variants (keys: prompt_variant, option_perm, subset, ids_file); None = the frozen v1 setting only.
    # One job and run name per variant; see the batch presets in docs/05-batch-runs.md
    "variants": None,
    "extra_args": [],  # flags for every job, e.g. ["--max-new-tokens", "32"] for the format check
    # Self-explanations: one job per *-fnfp.txt id list written by 99 (folder under OUTPUT_ROOT); None = cross product
    "ids_dir": None,
    "ids_prompt_variant": "X",
    "limit": None,  # first N rows, for debugging only (changes the run names)
    "seed": 42,
    # Shots per class from the frozen configs/fewshot/{dataset}/{task}_k{shots}_s{seed}.yaml; 0 = zero-shot
    "shots": 0,
    # TODO: think-on runs (Qwen3.5-4B) need a generate-then-score path; JSON-constrained decoding blocks thinking
    "thinking_models": [],
    # Parallel jobs on one GPU: packed by each model's vram_gb estimate (configs/llms.yaml); 1 = sequential
    "max_parallel": 7,
    "gpu_budget_gb": 70,  # of the A100's 80 GB; keep headroom for estimate error
    "heartbeat_s": 60,  # how often heartbeat.json on Drive is refreshed
    "dry_run": False,  # True: print the job table only
    "stop_on_error": False,  # sequential mode only
}
# Scheduling options may change between sessions without making a new experiment
SCHEDULING = ("max_parallel", "gpu_budget_gb", "heartbeat_s", "dry_run", "stop_on_error")
SAMPLE = CONFIG["sample"] is not None
print("experiment:", CONFIG["experiment"] + ("-sample" if SAMPLE else ""), "| sample:", CONFIG["sample"])

In [6]:
from src.utils import set_seeds

# Each job sets and logs its own seeds in run_inference.py; this covers the notebook process only
set_seeds(CONFIG["seed"])

In [ ]:
from src.prompts import freeze_shots, shots_path
from src.schemas import get_schema

# Few-shot sets: seeded, stratified train picks, frozen once (IDs only) and shared by every model; never overwritten
if CONFIG["shots"]:
    for dataset, task in CONFIG["tasks"]:
        path = shots_path(dataset, task, CONFIG["shots"], CONFIG["seed"])
        if not path.exists():
            freeze_shots(DATA_ROOT, get_schema(dataset, task), k=CONFIG["shots"], seed=CONFIG["seed"])
        print(path.relative_to(PROJECT_ROOT))


In [ ]:
# Token from Colab secrets (or the local env), passed to the job processes, never printed or pasted here
if IN_COLAB:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN set:", bool(HF_TOKEN), "(Llama 3.1 and Aya Expanse are gated and need it)")

import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.0f} GB")
else:
    print("no GPU: keep dry_run True here")

In [ ]:
from src.utils import (
    build_jobs, experiment_date, jobs_from_ids_dir, load_llm_registry, open_experiment, runs_dir, status_table,
)

# Opens the experiment folder, or reopens it if CONFIG (minus scheduling keys) is unchanged
EXP_DIR = open_experiment(
    OUTPUT_ROOT, CONFIG["experiment"], {k: v for k, v in CONFIG.items() if k not in ("purpose", "protocol_ref")},
    purpose=CONFIG["purpose"], protocol_ref=CONFIG["protocol_ref"], project_root=PROJECT_ROOT, sample=SAMPLE,
    ignore=SCHEDULING + (("sample",) if SAMPLE else ()),
)
RUNS = runs_dir(EXP_DIR)
REGISTRY = load_llm_registry(PROJECT_ROOT / "configs" / "llms.yaml")
MODELS = list(REGISTRY) if CONFIG["models"] == "all" else CONFIG["models"]
if CONFIG["ids_dir"]:  # one job per id list (models, tasks and splits come from the file names)
    JOBS = jobs_from_ids_dir(
        REGISTRY, OUTPUT_ROOT / CONFIG["ids_dir"], date=experiment_date(EXP_DIR), seed=CONFIG["seed"],
        backend=CONFIG["backend"], prompt_variant=CONFIG["ids_prompt_variant"], experiment_dir=str(EXP_DIR),
        sample=CONFIG["sample"], extra_args=tuple(CONFIG["extra_args"]),
    )
else:
    JOBS = build_jobs(
        REGISTRY, MODELS, CONFIG["tasks"], CONFIG["splits"],
        date=experiment_date(EXP_DIR), seed=CONFIG["seed"], backend=CONFIG["backend"], limit=CONFIG["limit"],
        thinking_models=tuple(CONFIG["thinking_models"]), shots=CONFIG["shots"],
        experiment_dir=str(EXP_DIR), sample=CONFIG["sample"], variants=CONFIG["variants"],
        extra_args=tuple(CONFIG["extra_args"]),
    )
before = status_table(RUNS, JOBS)
print(EXP_DIR)
print(f"{len(JOBS)} jobs:", before.status.value_counts().to_dict())
before.pivot_table(index="model", columns="status", values="run_name", aggfunc="count", fill_value=0)

In [ ]:
import json
from functools import partial

from src.utils import job_status, run_job, run_jobs_parallel, tail, write_heartbeat

SCRIPT = PROJECT_ROOT / "run_inference.py"
LOG_DIR = EXP_DIR / "logs"
ENV = {"DATA_ROOT": str(DATA_ROOT), "OUTPUT_ROOT": str(OUTPUT_ROOT), "HF_HOME": str(HF_HOME)}
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
beat = partial(write_heartbeat, EXP_DIR)

FAILED = {}
todo = [j for j in JOBS if job_status(RUNS, j)[0] in ("todo", "partial")]
if CONFIG["dry_run"]:
    print(f"dry run: {len(todo)} jobs would run; set dry_run False to launch")
elif CONFIG["max_parallel"] > 1:
    results = run_jobs_parallel(
        todo, SCRIPT, ENV, LOG_DIR, gpu_budget_gb=CONFIG["gpu_budget_gb"], max_parallel=CONFIG["max_parallel"],
        heartbeat=beat, heartbeat_every_s=CONFIG["heartbeat_s"],
    )
    FAILED = {name: tail(log) for name, (code, log) in results.items() if code != 0}
else:
    for k, job in enumerate(todo, 1):
        beat({"state": "running", "n_jobs": len(todo), "pending": len(todo) - k + 1, "running": [job.run_name],
              "done_ok": k - 1 - len(FAILED), "failed": sorted(FAILED)})
        print(f"[{k}/{len(todo)}] {job.run_name}")
        code, log = run_job(job, SCRIPT, ENV, LOG_DIR)
        if code != 0:
            FAILED[job.run_name] = tail(log)
            print("  FAILED:\n" + FAILED[job.run_name])
            if CONFIG["stop_on_error"]:
                break
    beat({"state": "finished", "n_jobs": len(todo), "done_ok": len(todo) - len(FAILED), "failed": sorted(FAILED)})
for name, msg in FAILED.items():
    print(f"FAILED {name}:\n{msg}\n")
print("failed:", list(FAILED) or "none")

In [ ]:
from src.utils import sanity_table, write_manifest

# Status record for this session; earlier sessions' tables are kept, never overwritten
stamp = f"{datetime.datetime.now():%Y%m%d-%H%M%S}"
after = status_table(RUNS, JOBS)
after.to_csv(EXP_DIR / f"status_{stamp}.csv", index=False)
(EXP_DIR / f"failed_{stamp}.json").write_text(json.dumps(FAILED, indent=2))
MANIFEST = write_manifest(EXP_DIR)  # rebuilt from the run folders; 99 reads runs only through it
print(after.status.value_counts().to_dict(), "| manifest rows:", len(MANIFEST))
if SAMPLE:
    SANITY = sanity_table(EXP_DIR, [j.run_name for j in JOBS if not j.blocked])
    print("SANITY OK" if SANITY.ok.all() else f"SANITY FAILED: {int((~SANITY.ok).sum())} of {len(SANITY)} runs")
    print("next: set CONFIG['sample'] = None and run all for the full batch" if SANITY.ok.all() else "see logs/")
    display(SANITY[~SANITY.ok] if not SANITY.ok.all() else SANITY.head())
else:
    display(after[after.status != "done"])

## Summary
TODO after the first full batch: number of finished / failed / blocked jobs per model, refusal-mass observations on dev, and anything that needed a rerun. Results and per-run metrics are in each run folder; this notebook only orchestrates.

In [13]:
import sys, importlib.metadata as md
print(sys.version.split()[0], {p: md.version(p) for p in ["numpy", "pandas", "pyarrow", "transformers", "torch"]})
try:
    import numpy; print("numpy imported in kernel:", numpy.__version__)
except Exception as e:
    print("numpy import error:", type(e).__name__, e)

3.13.15 {'numpy': '2.5.4', 'pandas': '3.0.6', 'pyarrow': '26.0.0', 'transformers': '5.20.1', 'torch': '2.11.0+cu130'}
numpy imported in kernel: 2.1.3


In [14]:
from pathlib import Path
root = Path("/content/drive/MyDrive/Sexism-research")
checks = {
    "configs/llms.yaml has vram_gb": "vram_gb" in (root / "configs/llms.yaml").read_text() if (root / "configs/llms.yaml").exists() else "missing",
    "jobs.py has run_jobs_parallel": "run_jobs_parallel" in (root / "src/utils/jobs.py").read_text() if (root / "src/utils/jobs.py").exists() else "missing",
    "task_1_2 template": (root / "src/prompts/templates/exist/task_1_2.yaml").exists(),
    "exist_dev has intention": None,
}
import pyarrow.parquet as pq
p = root / "data/processed/dev/exist_dev.parquet"
checks["exist_dev has intention"] = "intention" in pq.read_schema(p).names if p.exists() else "missing"
print(checks)

{'configs/llms.yaml has vram_gb': False, 'jobs.py has run_jobs_parallel': False, 'task_1_2 template': True, 'exist_dev has intention': True}
